In [ ]:
import os
from os.path import isfile, join
import glob
from pydicom import dcmread
from pydicom.multival import MultiValue
import re
import numpy as np
from utils import remove_overlap
import warnings
from scipy.ndimage import zoom
import matplotlib.pyplot as plt

"""
Discard reasons:
Left
ANON1832: No dose, rtstruct, ct, or mask
ANON1845, ANON1861, ANON0090, ANON0091, ANON1792, ANON0132: Missing critical structures (PTV, heart, lungs or contra breast)
"""

def downsample_dicom_folder(dataset):
    if dataset == 'L':
        DISCARD_LIST = ['ANON1832', 'ANON1845', 'ANON1861', 'ANON0090', 'ANON0091', 'ANON1792', 'ANON0132']
        BEAMS = '.+2\.beams'
        SOURCE_PATH = "/home/akselile/Documents/DL/datasets/Breast_L/**/"
        DESTINATION_PATH = "/home/akselile/Documents/DL/datasets/Breast_L_ds/"
    elif dataset == 'R':
        DISCARD_LIST = ['']
        BEAMS = '.+2\.beams'
        SOURCE_PATH = "/home/akselile/Documents/DL/datasets/Breast_R/**/"
        DESTINATION_PATH = "/home/akselile/Documents/DL/datasets/Breast_R_ds/"
    elif dataset == 'LAX':
        DISCARD_LIST = ['ANON0941']
        BEAMS = '.+1\.beam'
        SOURCE_PATH = "/home/akselile/Documents/DL/datasets/BreastAx_L/**/"
        DESTINATION_PATH = "/home/akselile/Documents/DL/datasets/BreastAx_L_ds/"
    elif dataset == 'RAX':
        DISCARD_LIST = []
        BEAMS = '.+1\.beam'
        SOURCE_PATH = "/home/akselile/Documents/DL/datasets/BreastAx_R/**/"
        DESTINATION_PATH = "/home/akselile/Documents/DL/datasets/BreastAx_R_ds/"
        
    all_items = glob.glob(SOURCE_PATH, recursive=True)
    study_folders = [s for s in all_items if 'Anon' in s]
    r = re.compile('ANON\d{4}')
    unique_subjects = [r.search(s)[0] for s in study_folders if r.search(s)]
    unique_subjects = np.unique(unique_subjects)
    for i, subject in enumerate(unique_subjects):
        if not subject in DISCARD_LIST: # These are 50Gy or SIB, thus discarding
            r = re.compile(f"{subject}(.+RTPLAN|{BEAMS}){'.+00000'}")
            subject_folders = [s for s in study_folders if r.search(s)]

            # There should be CT, mask "CT", dose, and RTstruct for each subject.
            if len(subject_folders) == 0:
                pass
            else:
                if len(subject_folders) != 5:
                    warnings.warn("Subject " + subject + " has != 5 folders. Ensure manually that ct, dose, and mask are present.")
                
                r_ct = re.compile('(?!.*Mask.*Mask).+CT.+CT.+DVH.+CT')
                r_mask = re.compile('.+Mask.+Mask')
                r_dose = re.compile('.+Dose')
                r_plan = re.compile('.+RTPLAN')

                ct_path = [s for s in subject_folders if r_ct.search(s)]
                mask_path = [s for s in subject_folders if r_mask.search(s)]
                dose_path = [s for s in subject_folders if r_dose.search(s)]
                plan_path = [s for s in subject_folders if r_plan.search(s)]
                
                dcm_path_ct = os.listdir(ct_path[0])
                dcmfile_path_dose = dose_path[0] + os.listdir(dose_path[0])[0]
                dcm_path_mask = os.listdir(mask_path[0])
                dcmfile_path_plan = plan_path[0] + os.listdir(plan_path[0])[0]
                
                ### Ensin tehdään potilaskansio
                try:
                    os.mkdir(DESTINATION_PATH + subject)
                except FileExistsError:
                    pass
                
                ### Tallennetaan dose ekana. Tämä hieman erilainen kuin muut, koska kaikki leikkeet samassa filessä.
                dcm_dose = dcmread(dcmfile_path_dose)
                # Downsamplaus - harkitse filtteröintiä jos tätä suurempi kerroin
                dcm_dose_downsampled = dcm_dose
                #dcm_dose_downsampled = zoom(dcm_dose.pixel_array, zoom=(1, 0.5, 0.5), order=1)

                if dataset == 'RAX':
                    dose_copy = dcm_dose_downsampled.copy()
                
                # Päivitetään DICOMin muotokentät
                _, dcm_dose.Rows, dcm_dose.Columns = dcm_dose_downsampled.shape
                
                # Päivitettään myös pixel spacing, jotta pikselin koko edelleen oikein. Tuo spacing on decimal string (ds) ja sen tyyppi on
                # pydicomin MultiValue, siksi muutettu hieman hassusti.
                new_spacing = [float(x)*2 for x in list(dcm_dose.PixelSpacing)]
                dcm_dose.PixelSpacing = MultiValue(float, new_spacing)

                try:
                    os.mkdir(DESTINATION_PATH + subject + "/dose/")
                except FileExistsError:
                    pass
                
                
                ### Sitten CT
                try:
                    os.mkdir(DESTINATION_PATH + subject + "/ct/")
                except FileExistsError:
                    pass
                
                # Tässä joudutaan iteroimaan läpi koko kansio ja avataan ja tallennetaan yksittäiset leikkeet yksitellen.
                for dcmfile_ct in dcm_path_ct:
                    
                        
                    dcm_ct = dcmread(ct_path[0] + dcmfile_ct)
                    dcm_ct_downsampled = zoom(dcm_ct.pixel_array, zoom=(0.5, 0.5), order=1)
                    dcm_ct.PixelData = dcm_ct_downsampled.tobytes()


                    dcm_ct.Rows, dcm_ct.Columns = dcm_ct_downsampled.shape
                    new_spacing = [float(x)*2 for x in list(dcm_ct.PixelSpacing)]
                    dcm_ct.PixelSpacing = MultiValue(float, new_spacing)
                    dcm_ct.save_as(DESTINATION_PATH + subject + "/ct/" + dcmfile_ct)
                
                ### Maskit täysin samalla tavalla kuin CT
                try:
                    os.mkdir(DESTINATION_PATH + subject + "/mask/")
                except FileExistsError:
                    pass
                
                for dcmfile_mask in dcm_path_mask:
                    dcm_mask = dcmread(mask_path[0] + dcmfile_mask)
                    dcm_mask_downsampled = dcm_mask
                    #dcm_mask_downsampled = zoom(dcm_mask.pixel_array, zoom=(0.5, 0.5), order=0)
                    dcm_mask.PixelData = dcm_mask_downsampled.tobytes()

                    # Instance number is basically slice number, reversed compared to index of dose array
                    instance_number = dcm_mask.InstanceNumber
                    instance_number_reversed = np.shape(dcm_dose_downsampled)[0] - instance_number
                    
                    dcm_mask.Rows, dcm_mask.Columns = dcm_mask_downsampled.shape
                    mask_min = np.min(dcm_mask_downsampled)
                    
                    dcm_dose_downsampled[instance_number_reversed, :, :] = np.multiply(dcm_dose_downsampled[instance_number_reversed, :, :], np.isin(dcm_mask_downsampled, mask_min, invert = True)) #1023 equals to -1

                    new_spacing = [float(x)*2 for x in list(dcm_mask.PixelSpacing)]
                    dcm_mask.PixelSpacing = MultiValue(float, new_spacing)
                    dcm_mask.save_as(DESTINATION_PATH + subject + "/mask/" + dcmfile_mask)
    
                # Haetaan tallennetaan pienennetty data takaisin DICOMiin
                dcm_dose.PixelData = dcm_dose_downsampled.tobytes()
                dcm_dose.save_as(DESTINATION_PATH + subject + "/dose/" + os.listdir(dose_path[0])[0])

                ### Lopuksi RTPLANit
                try:
                    os.mkdir(DESTINATION_PATH + subject + "/plan/")
                except FileExistsError:
                    pass

                dcm_plan = dcmread(dcmfile_path_plan)
                dcm_plan.save_as(DESTINATION_PATH + subject + "/plan/" + os.listdir(plan_path[0])[0])

print("PROCESSING...")
downsample_dicom_folder('R')
print("DONE")

In [ ]:
subject_folders

In [ ]:
np.any(np.isin([1, 2, 3, 4, 5], [1, 2]))